# 13 — Type Hints and Static Typing (`typing`)

Goal: use type hints to catch bugs earlier, document APIs, and scale codebases.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
python -m pip install -U mypy pyright
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Why type hints?

Type hints:
- improve editor autocomplete
- act as documentation
- enable static analysis (mypy/pyright)
- reduce runtime bugs in large codebases

Python remains dynamically typed at runtime unless you enforce checks yourself.

## 2.
L2: Core syntax (Python 3.9+)

- `list[int]`, `dict[str, int]`
- `X | None` instead of `Optional[X]`

In [ ]:

def first(xs: list[int]) -> int | None:
    return xs[0] if xs else None

print(first([1,2,3]))
print(first([]))


## 3.
L3: Protocols (structural typing)

A Protocol defines behavior by required methods/attributes.
Any object matching the shape satisfies the type checker.

In [ ]:

from typing import Protocol

class SupportsClose(Protocol):
    def close(self) -> None: ...

def close_all(items: list[SupportsClose]) -> None:
    for x in items:
        x.close()

class Thing:
    def close(self) -> None:
        print("closed")

close_all([Thing(), Thing()])


## 4.
L4: TypedDict (dicts with known keys)

Useful for JSON-like objects when you don’t want a full dataclass/model.

In [ ]:

from typing import TypedDict

class UserRow(TypedDict):
    id: int
    name: str
    active: bool

u: UserRow = {"id": 1, "name": "Ada", "active": True}
print(u["name"])


## 5.
L5: Generics with TypeVar

Use generics to preserve types across operations.

In [ ]:

from typing import TypeVar

T = TypeVar("T")

def head(xs: list[T]) -> T:
    return xs[0]

print(head(["a","b"]))
print(head([10, 20]))


## 6.
L6: Overloads (for precise APIs)

`@overload` lets you describe multiple call signatures for one implementation.
Runtime sees only the final implementation.

In [ ]:

from typing import overload

@overload
def parse(x: str) -> int: ...
@overload
def parse(x: int) -> str: ...

def parse(x):
    if isinstance(x, str):
        return int(x)
    return str(x)

print(parse("12"))
print(parse(12))


## 7.
L7: Type-checking workflow

Typical project commands:

```bash
ruff check .
black .
pytest
mypy src
# or:
pyright
```

## 8.
L8: Exercises

1. Write `unique(xs: list[T]) -> list[T]` preserving order.
2. Define a `Protocol` for objects that have `__len__`.
3. Create a `TypedDict` for a simple API response.

In [ ]:

from typing import TypeVar

T = TypeVar("T")

def unique(xs: list[T]) -> list[T]:
    seen: set[T] = set()
    out: list[T] = []
    for x in xs:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

assert unique([1,1,2,3,2]) == [1,2,3]
assert unique(list("abac")) == ["a","b","c"]
print("ok")


## 9.
L9: Literal types and `Final`

- `Literal[...]` constrains to specific values
- `Final` indicates “should not be reassigned”

In [ ]:

from typing import Literal, Final

Mode = Literal["dev", "prod"]
DEFAULT_MODE: Final[Mode] = "dev"

def run(mode: Mode) -> str:
    return f"mode={mode}"

print(run("dev"))


## 10.
L10: Type narrowing, `cast`, and `TypeGuard`

Type checkers can narrow types with `isinstance` checks.
`cast` tells the type checker “trust me” (no runtime effect).

In [ ]:

from typing import cast

def maybe_str(x: object) -> str | None:
    if isinstance(x, str):
        return x
    return None

v = maybe_str("hi")
if v is not None:
    # narrowed to str
    print(v.upper())

# cast example (use rarely)
y = cast(str, "hello")
print(y)


## 11.
L11: `dataclass` + typing

Type hints on dataclasses help mypy/pyright and IDEs.
Combine with `slots=True` for performance in large data models.